<a href="https://colab.research.google.com/github/askarbekkk/kyrgyz-embeddings/blob/main/kyrgyz-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets -q
from datasets import load_dataset
import random

ds = load_dataset("Zhantas/Cleaned-Kyrgyz_Wikipedia", split="train")

chunks = []

for art in ds.select(range(5000)):
  for para in art["text"].split("\n"):
    if 200 < len(para) < 800:
      chunks.append({"title": art["title"], "text": para.strip()})

print(len(chunks))
random.seed(42)

for c in random.sample(chunks, 3):
  print(c["title"], "|", c["text"][:200], "\n")



README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 67.2MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/76519 [00:00<?, ? examples/s]

14250
Саймалыташ сүрөт галереясы | Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан. Кийин аны генерал –мойор И.Т. Пославский (1902-1903), Б.М.Зима (1946), А.Н.Бернштам(1950) изилдеген. Саймалыта 

Кыргыз тилинин грамматикасы | Баяндагыч ыңгай: Кыймыл-аракеттин ошол учурда болуп өткөнүн, болуп жатканын, боло турганын жайынча баяндаган этиш сөздөр баяндагыч ыңгай деп аталат. Мисалы: Шаарга бардым. Шаарга бара жатам. Шаарга ба 

Айыл чарба | Асыл тукум мал чарбасын 228 чарба жүргүзүүчү субъект түзөт. Айыл чарбаны пландоо жана кеңеш берүү жактан камсыз кылуу үчүн агрардык илим жана консулътациялык кызмат борбору, анын карамагындагы төрт ил 



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')


In [ ]:
!pip install openai -q

from openai import OpenAI
from google.colab import userdata, drive
import random, time, json, re, os

drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/pairs_raw_v2.json"

client = OpenAI(
    api_key=userdata.get('openrouter_key'),
    base_url="https://openrouter.ai/api/v1",
)
MODEL = "openai/gpt-oss-120b"

PROMPT = """You are given a passage in the Kyrgyz language.

Write ONE short question in Kyrgyz that can be answered using this passage.
The question must sound like something a real person would type into a search engine — natural, specific, and self-contained.

Rules:
- Write the question in Kyrgyz only.
- Do not copy full sentences from the passage.
- Do not reference "the text" or "the passage".
- Return only the question, with no explanation or extra formatting.

Passage:
{text}"""

def clean_text(t):
    """Strip leftover MediaWiki markup and normalize whitespace."""
    t = re.sub(r'thumb\|[^ ]*', '', t)
    t = re.sub(r'\b\d+px\b', '', t)
    t = re.sub(r'(center|right|left)\|', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

random.seed(42)
sample = random.sample(chunks, 2500)

# Resume from a previous run: skip passages already processed
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, encoding="utf-8") as f:
        pairs = json.load(f)
    done = {p["positive"] for p in pairs}
    print(f"Loaded {len(pairs)} existing pairs, resuming")
else:
    pairs, done = [], set()
    print("Starting from scratch")

def save():
    """Write atomically so an interrupted run cannot corrupt the file."""
    tmp = SAVE_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)
    os.replace(tmp, SAVE_PATH)

try:
    for i, c in enumerate(sample):
        text = clean_text(c["text"])
        if len(text) < 200 or text in done:
            continue

        for attempt in range(3):
            try:
                r = client.chat.completions.create(
                    model=MODEL,
                    messages=[{"role": "user", "content": PROMPT.format(text=text)}],
                )
                pairs.append({
                    "query": r.choices[0].message.content.strip(),
                    "positive": text,
                    "title": c["title"],
                })
                done.add(text)
                break
            except Exception as e:
                print(f"[{i}] attempt {attempt+1}: {e}")
                time.sleep(5 * (attempt + 1))  # back off before retrying

        if len(pairs) % 25 == 0:
            save()
            print(f"i={i}, collected {len(pairs)}, saved")
        time.sleep(0.5)

except KeyboardInterrupt:
    print("Interrupted by user")

finally:
    save()  # always persist, even on crash or manual stop
    print(f"\nTOTAL: {len(pairs)} pairs in {SAVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 1650 existing pairs, resuming
i=1708, collected 1675, saved
i=1733, collected 1700, saved
Interrupted by user



KeyboardInterrupt



In [ ]:
c = sample[0]
r = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=PROMPT.format(text=c["text"]),
)
print("QUERY:", r.text.strip())
print("\nTEXT:", c["text"][:300])

In [ ]:
!ls -la /content/*.json
!ls -la /content/drive/MyDrive/*.json

In [ ]:
  # from google.colab import drive
  # drive.mount('/content/drive')

  # import json
  # with open("/content/drive/MyDrive/pairs_raw.json", "w", encoding="utf-8") as f:
  #     json.dump(pairs, f, ensure_ascii=False, indent=2)
  # print("saved", len(pairs))
  !ls -la /content/drive/MyDrive/pairs_raw_v2.json

-rw------- 1 root root 1524777 Sep  8 23:28 /content/drive/MyDrive/pairs_raw_v2.json


In [ ]:
!rm /content/drive/MyDrive/pairs_clean_v2.json

In [ ]:
!pip install google-genai groq -q

import json, time, os, re, threading
from concurrent.futures import ThreadPoolExecutor
from google import genai
from groq import Groq
from google.colab import userdata

gemini = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
groq   = Groq(api_key=userdata.get('groq_apikey'))

# Tried in order; each entry falls back to the next when its quota runs out
BACKENDS = [
    ("gemini", "gemini-3.5-flash-lite"),
    ("gemini", "gemini-3.1-flash-lite"),
    ("gemini", "gemini-3.6-flash"),
    ("groq",   "openai/gpt-oss-20b"),
    ("groq",   "openai/gpt-oss-120b"),
]

RAW_PATH   = "/content/drive/MyDrive/pairs_raw_v2.json"
CLEAN_PATH = "/content/drive/MyDrive/pairs_clean_v2.json"

CHECK = """You are given a passage in Kyrgyz and a question in Kyrgyz.

Answer YES only if ALL of these hold:
1. The question is written in Kyrgyz.
2. The passage contains a complete answer to the question.
3. The question is specific — not a vague or generic prompt.
4. The question makes sense on its own, without seeing the passage.

Otherwise answer NO.

Reply with exactly one word: YES or NO.

Passage:
{text}

Question:
{query}"""

def is_bibliography(t):
    """Reject citation lists and reference entries — not usable prose."""
    return bool(re.search(r'//|Тез\. докл\.|\bС\.\s*\d+-\d+', t)) or t.count('.') > len(t) / 30

with open(RAW_PATH, encoding="utf-8") as f:
    pairs = json.load(f)
print(f"loaded {len(pairs)} raw pairs")

if os.path.exists(CLEAN_PATH):
    with open(CLEAN_PATH, encoding="utf-8") as f:
        clean = json.load(f)
    checked = {p["query"] for p in clean}
    print(f"resuming, {len(clean)} already kept")
else:
    clean, checked = [], set()

lock = threading.Lock()
errors = []
exhausted = set()          # backends whose quota is gone for today
usage = {}                 # how many calls each backend served

def save():
    """Write atomically so an interrupted run cannot corrupt the file."""
    with lock:
        data = list(clean)
    tmp = CLEAN_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, CLEAN_PATH)

def ask(provider, model, prompt):
    if provider == "gemini":
        r = gemini.models.generate_content(model=model, contents=prompt)
        return r.text
    r = groq.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
    )
    return r.choices[0].message.content

def is_quota_error(e):
    s = str(e).lower()
    return "429" in s or "resource_exhausted" in s or "rate limit" in s or "quota" in s

def check_one(p):
    prompt = CHECK.format(text=p["positive"], query=p["query"])
    for provider, model in BACKENDS:
        if (provider, model) in exhausted:
            continue
        for attempt in range(2):
            try:
                text = ask(provider, model, prompt)
                if text and "YES" in text.strip().upper():
                    with lock:
                        clean.append(p)
                with lock:
                    usage[model] = usage.get(model, 0) + 1
                return
            except Exception as e:
                if is_quota_error(e):
                    with lock:
                        if (provider, model) not in exhausted:
                            exhausted.add((provider, model))
                            print(f"  quota exhausted: {model}")
                    break                      # move to next backend
                time.sleep(3 * (attempt + 1))  # transient error, retry same backend
    with lock:
        errors.append(p["query"][:60])

todo = [p for p in pairs
        if p["query"] not in checked and not is_bibliography(p["positive"])]
print(f"to check: {len(todo)}")

with ThreadPoolExecutor(max_workers=3) as ex:
    for n, _ in enumerate(ex.map(check_one, todo)):
        if n % 50 == 0:
            save()
            print(f"{n}/{len(todo)}, kept {len(clean)}, errors {len(errors)}, usage {usage}")

save()
print(f"\n{len(clean)} of {len(pairs)} passed, {len(errors)} failed")
print("usage:", usage)

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": CHECK.format(text=p["positive"], query=p["query"])}],
    max_tokens=512,
)
print(repr(r.choices[0].message.content))
print(repr(getattr(r.choices[0].message, "reasoning", None)))

'YES'
'We need to evaluate conditions.\n\n1. Question is in Kyrgyz: "Кайсы орус армиясынын офицери 1902-жылы Саймалыташ сүрөт галереясын ачты?" This is Kyrgyz. Yes.\n\n2. Passage contains a complete answer to the question. The passage says: "Саймалыташ сүрөт галереясын 1902-жылы орус армиясынын офицери, топограф Н.Г.Хлудов ачкан." So answer: Н.Г.Хлудов. That\'s a complete answer. Yes.\n\n3. Question is specific — not vague. It asks which officer opened the gallery in 1902. Specific.\n\n4. Question makes sense on its own, without seeing passage. Yes.\n\nThus all conditions hold, answer YES.'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls -la /content/drive/MyDrive/*.json

Mounted at /content/drive
-rw------- 1 root root  250543 Sep  9 03:03 /content/drive/MyDrive/pairs_clean.json
-rw------- 1 root root       2 Sep  9 03:08 /content/drive/MyDrive/pairs_clean_v2.json
-rw------- 1 root root    1458 Sep  8 14:40 /content/drive/MyDrive/pairs_raw.json
-rw------- 1 root root 1524777 Sep  8 23:28 /content/drive/MyDrive/pairs_raw_v2.json
-rw------- 1 root root    1519 Sep  8 14:29 /content/drive/MyDrive/results.json


In [ ]:
import json

with open("/content/drive/MyDrive/pairs_clean_v2.json", encoding="utf-8") as f:
        clean = json.load(f)


with open("/content/drive/MyDrive/pairs_raw_v2.json", encoding="utf-8")  as f:
       pairs = json.load(f)

print(len(clean), len(pairs))


482 1700


In [ ]:
kept_q = {p["query"] for p in clean}
rejected = [p for p in pairs if p["query"] not in kept_q]

import random
random.seed(1)
for p in random.sample(rejected, 10):
    print("Q:", p["query"])
    print("T:", p["positive"][:200])
    print("---")

Q: Чакатамар үңкүрүндө Аятолл курсынын үзүндү барбы?
T: Экинчиси – Чакатамар. «Чакатамар» сөзү “тамчы” дегенди билдирет. Бул үңкүр барган адам үчүн таңкалычтуу, сүрдүү. Ал жерге кирчү жол өтө кууш. Бир адам батчу тар жол менен сойлоп киресиң. Шам жаксаң ич
---
Q: Таза жеңиш жок болгон учурда жеңүүчү команда кандай критерийлерге жараша аныкталат?
T: Жеңиштердин саны тең болгондо бардык таза жеңиштерге убакытты эң аз жумшаган топ алдыда деп эсептелинет. Бул эки топтордун беттешүүсүндө бир таза жеңиш болбогондо жеңип чыккан кармашууларда утуш упайы
---
Q: Кыркчоронун ордосу кайда жайгашкан?
T: Кыркчоронун ордосу – Кочкордун түндүк- батыш тарабындагы Каратоонун үстүндөгү түздүк. Узун- туурасы бир гадай болгон ачык жай, ордо ойногондой тегерек, ортосунда кан жайгашкандай кичинекей белгиси бар
---
Q: Салмоор ыргытуучу жана тургуча жердин аралыгы канча метр?
T: Салмоор ыргытуу боюнча мелдеш уюштурулат. Калыс оюнчуларды эки топко бөлөт. Салмоорду ыргытуучу, салмоор түшүүчү, оюнчулар турчу жер 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -U sentence-transformers -q

import json, random
from sentence_transformers import (
    SentenceTransformer, losses,
    SentenceTransformerTrainer, SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from datasets import Dataset

# --- data ---
with open("/content/drive/MyDrive/pairs_raw_v2.json", encoding="utf-8") as f:
    data = json.load(f)

seen, uniq = set(), []
for p in data:
    if p["positive"] not in seen:
        seen.add(p["positive"])
        uniq.append(p)
print(f"{len(uniq)} unique pairs")

random.seed(42)
random.shuffle(uniq)
split = int(len(uniq) * 0.9)
train_pairs, test_pairs = uniq[:split], uniq[split:]
print(len(train_pairs), "train /", len(test_pairs), "test")

# --- evaluation setup: search gold passages among 5000 distractors ---
corpus = {str(i): c["text"] for i, c in enumerate(chunks[:5000])}
queries, relevant = {}, {}
for qi, p in enumerate(test_pairs):
    doc_id = f"gold_{qi}"
    corpus[doc_id] = p["positive"]
    queries[str(qi)] = p["query"]
    relevant[str(qi)] = {doc_id}

evaluator = InformationRetrievalEvaluator(
    queries=queries, corpus=corpus, relevant_docs=relevant, name="ky-v2"
)
print(f"corpus {len(corpus)}, queries {len(queries)}")

# --- baseline ---
model = SentenceTransformer("intfloat/multilingual-e5-base")
before = evaluator(model)
print("BEFORE acc@1:", before['ky-v2_cosine_accuracy@1'])
print("BEFORE nDCG@10:", before['ky-v2_cosine_ndcg@10'])

# --- training ---
train_ds = Dataset.from_dict({
    "anchor":   [p["query"]    for p in train_pairs],
    "positive": [p["positive"] for p in train_pairs],
})

args = SentenceTransformerTrainingArguments(
    output_dir="/content/ky-e5-base-v2",
    num_train_epochs=3,
    per_device_train_batch_size=64,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=5,
)

SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    loss=losses.MultipleNegativesRankingLoss(model),
).train()

after = evaluator(model)
print("\nAFTER acc@1:", after['ky-v2_cosine_accuracy@1'])
print("AFTER nDCG@10:", after['ky-v2_cosine_ndcg@10'])

# --- save ---
model.save("/content/drive/MyDrive/ky-e5-base-v2")
with open("/content/drive/MyDrive/results_v2.json", "w") as f:
    json.dump({"before": before, "after": after,
               "n_train": len(train_pairs), "n_test": len(test_pairs)}, f, indent=2)

1698 unique pairs
1528 train / 170 test
corpus 5170, queries 170


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
5,2.389043
10,0.915324
15,0.327161
20,0.106487
25,0.124217
30,0.047219
35,0.065348
40,0.042279
45,0.046520
50,0.033597


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


AFTER acc@1: 0.6352941176470588
AFTER nDCG@10: 0.8405313204298904


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
after = evaluator(model)
print("BEFORE acc@1:", before['ky-v2_cosine_accuracy@1'])
print("AFTER  acc@1:", after['ky-v2_cosine_accuracy@1'])
print("BEFORE nDCG@10:", before['ky-v2_cosine_ndcg@10'])
print("AFTER  nDCG@10:", after['ky-v2_cosine_ndcg@10'])

BEFORE acc@1: 0.6176470588235294
AFTER  acc@1: 0.6352941176470588
BEFORE nDCG@10: 0.820853599212609
AFTER  nDCG@10: 0.8405313204298904


In [7]:
model.save("/content/drive/MyDrive/ky-e5-base-v2")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluates a classical BM25 lexical baseline on the same 170 test queries and 5,170-document corpus, so its retrieval quality can be compared directly against the neural embedding models.

In [8]:
!pip install rank_bm25 -q

from rank_bm25 import BM25Okapi
import re, numpy as np

# same corpus as the neural evaluation
doc_ids = list(corpus.keys())
docs    = [corpus[d] for d in doc_ids]

def tok(t):
    return re.findall(r'\w+', t.lower())

bm25 = BM25Okapi([tok(d) for d in docs])

hits1, ndcg = 0, []
for qid, q in queries.items():
    scores = bm25.get_scores(tok(q))
    top10  = np.argsort(scores)[::-1][:10]
    gold   = relevant[qid]

    ranked = [doc_ids[i] for i in top10]
    if ranked[0] in gold:
        hits1 += 1

    # nDCG@10 with a single relevant document
    dcg = 0.0
    for rank, d in enumerate(ranked, start=1):
        if d in gold:
            dcg = 1 / np.log2(rank + 1)
            break
    ndcg.append(dcg)

print(f"BM25 acc@1:   {hits1/len(queries):.4f}")
print(f"BM25 nDCG@10: {np.mean(ndcg):.4f}")
print()
print("e5-base baseline:  0.618 / 0.821")
print("e5-base fine-tuned: 0.641 / 0.843")

BM25 acc@1:   0.6235
BM25 nDCG@10: 0.7695

e5-base baseline:  0.618 / 0.821
e5-base fine-tuned: 0.641 / 0.843
